# 🔴 Solution: Implement LayerNorm（参考版）

## 📋 题目描述

**难度：** Medium

实现层归一化。

LayerNorm 对每个样本沿特征维度进行归一化，不依赖批大小即可稳定训练。

**签名:** `my_layer_norm(x, gamma, beta, eps=1e-5) -> Tensor`

**参数:**
- `x` — 输入张量 (..., D)
- `gamma` — 缩放参数 (D,)
- `beta` — 偏移参数 (D,)
- `eps` — 数值稳定性的 epsilon

**返回:** 归一化后的张量，形状与 x 相同

**约束:**
- 沿最后一个维度归一化
- 方差使用 `unbiased=False`
- 必须与 `F.layer_norm` 一致

**提示：** 1. `mean = x.mean(dim=-1, keepdim=True)`
2. `var = x.var(dim=-1, keepdim=True, unbiased=False)`
3. `x_norm = (x - mean) / sqrt(var + eps)` → `gamma * x_norm + beta`


In [ ]:
import torch
import math

In [ ]:
# ✅ SOLUTION

def layernorm(x: torch.Tensor, weight: torch.Tensor, bias: torch.Tensor, eps: float = 1e-5) -> torch.Tensor:
    # Layer Norm 对每个样本的最后几个维度做归一化
    # 公式：y = γ * (x - μ) / √(σ² + ε) + β
    # 其中 μ, σ² 是沿归一化维度的均值和方差

    # ---- Step 1: 计算均值 ----
    # keepdim=True 保持维度，便于后续广播
    # 例如 x shape=[32, 768]，沿 dim=-1 → mean shape=[32, 1]
    mean = x.mean(dim=-1, keepdim=True)

    # ---- Step 2: 计算方差 ----
    # Var(x) = E[(x - μ)²] = E[x²] - μ²
    # 这里用 (x - mean)² 的均值，即有偏方差（PyTorch 默认 behavior）
    var = ((x - mean) ** 2).mean(dim=-1, keepdim=True)

    # ---- Step 3: 归一化 ----
    # 减均值除标准差，得到均值为0方差为1的分布
    # eps=1e-5 防止除零（当方差接近0时）
    x_norm = (x - mean) / torch.sqrt(var + eps)

    # ---- Step 4: 仿射变换（可学习参数） ----
    # γ (weight) 和 β (bias) 是可学习的缩放和偏移参数
    # 让网络可以"撤销"归一化（如果需要的话）
    # weight, bias shape 通常为 [768]，广播到 [32, 768]
    return weight * x_norm + bias

In [ ]:
# Verify
x = torch.randn(2, 8)
gamma = torch.ones(8)
beta = torch.zeros(8)
out = my_layer_norm(x, gamma, beta)
ref = torch.nn.functional.layer_norm(x, [8], gamma, beta)
print("Match ref?", torch.allclose(out, ref, atol=1e-4))

In [ ]:
# Run judge
from torch_judge import check
check("layernorm")

## 📝 核心思路总结

1. **归一化公式**：(x - μ) / √(σ² + ε)，沿最后维度计算统计量
2. **keepdim 的必要性**：保持维度用于广播，避免形状错误
3. **仿射变换**：γ, β 可学习参数，让网络恢复表达能力
4. **eps 的作用**：数值稳定，防止除零